In [24]:
"""
Konverter Data Real Siswa SMK - Advanced Feature Engineering
============================================================
Pembaruan:
1. Menambahkan fitur 'Jurusan'
2. Menghitung 'SES_Index' (Socio-Economic Status)
3. Menghitung 'Deviasi_Tidur' (Penyimpangan dari 8 jam)
4. Menstandarisasi 'Nilai_Raport' menjadi Z-Score per Jurusan
"""

import pandas as pd
import numpy as np
import re
import os

SCRIPT_DIR = os.path.dirname(os.path.abspath('normalisasi_dan_konverter_data_real.ipynb'))
INPUT_FILE = os.path.join(SCRIPT_DIR, 'ANALISIS PENGARUH FAKTOR GAYA HIDUP DAN KEBIASAAN BELAJAR TERHADAP KINERJA AKADEMIK SISWA SMK.csv')
OUTPUT_FILE = os.path.join(SCRIPT_DIR, 'data_advanced.csv')

# ============================================================
# 1. BACA DATA
# ============================================================
for enc in ['utf-8-sig', 'utf-8', 'latin-1']:
    try:
        with open(INPUT_FILE, 'r', encoding=enc) as f:
            first_line = f.readline()
        break
    except:
        enc = 'utf-8'

sep = ';' if first_line.count(';') > first_line.count(',') else ','
df = pd.read_csv(INPUT_FILE, sep=sep, encoding=enc)

In [25]:
# ============================================================
# 2. DETEKSI KOLOM
# ============================================================
col_map = {}
for col in df.columns:
    cl = col.lower()
    if 'nisn' in cl: col_map['nisn'] = col
    elif 'nama lengkap' in cl or 'nama siswa' in cl: col_map['nama'] = col
    elif 'email' in cl: col_map['email'] = col
    elif 'jenis kelamin' in cl or 'gender' in cl: col_map['gender'] = col
    elif 'jurusan' in cl: col_map['jurusan'] = col
    elif ('avg' in cl or 'rata' in cl) and 'raport' in cl: col_map['nilai_raport'] = col
    elif 'kehadiran' in cl and 'pelatihan' not in cl and 'industri' not in cl and 'industry' not in cl: col_map['kehadiran'] = col
    elif 'pendidikan terakhir ayah' in cl: col_map['pend_ayah'] = col
    elif 'pendidikan terakhir ibu' in cl: col_map['pend_ibu'] = col
    elif 'pemasukan keluarga' in cl or 'pendapatan' in cl: col_map['pendapatan'] = col
    elif 'pekerjaan sampingan' in cl: col_map['kerja_sampingan'] = col
    elif 'waktu yang kamu habiskan untuk belajar' in cl: col_map['jam_belajar'] = col
    elif 'jam tidur' in cl or 'tidur per malam' in cl: col_map['jam_tidur'] = col
    elif 'bermain hp' in cl or 'screen' in cl: col_map['screen_time'] = col
    elif 'masalah aneh' in cl: col_map['q9'] = col
    elif 'supervisor' in cl or 'aplikasi atau alat brand baru' in cl: col_map['q10'] = col
    elif 'kemampuan keahlian praktik' in cl: col_map['kompetensi'] = col
    elif 'konsentrasi penuh' in cl: col_map['q11'] = col
    elif 'posisi hp' in cl or 'smartphone' in cl: col_map['q12'] = col
    elif 'ujian teori besok' in cl: col_map['q13'] = col
    elif 'grup whatsapp' in cl or 'mabar' in cl: col_map['q14'] = col
    elif 'titik jenuh' in cl or 'pikiran realistis' in cl: col_map['q15'] = col
    elif 'fisik sudah cukup lelah' in cl or 'refleks pertamamu' in cl: col_map['q16'] = col
    elif 'kritikan tajam' in cl or 'omelan' in cl: col_map['q17'] = col
    elif 'teman-teman di sekitarmu' in cl or 'suara batin' in cl: col_map['q18'] = col

In [26]:
# ============================================================
# 3. FUNGSI CLEANING
# ============================================================
def clean_num(val):
    if isinstance(val, str):
        val = val.replace(',', '.').replace('%', '').strip()
        val = re.sub(r'\s+', '', val)
        try: return float(val)
        except: return np.nan
    return float(val) if pd.notna(val) else np.nan

def is_hhmm_format(val):
    if pd.isna(val): return False
    try:
        s = str(int(val)).strip()
        if re.match(r'^\d{3,4}$', s):
            num = int(s)
            if 0 <= (num // 100) <= 23 and 0 <= (num % 100) <= 59: return True
    except: pass
    return False

# Clean Num Columns
for c in ['jam_tidur', 'screen_time', 'jam_belajar']:
    df[col_map[c]] = df[col_map[c]].apply(clean_num)

mask_hhmm = df[col_map['jam_tidur']].apply(is_hhmm_format)
df.loc[mask_hhmm | (df[col_map['jam_tidur']] > 12), col_map['jam_tidur']] = np.nan
df.loc[df[col_map['screen_time']] > 12, col_map['screen_time']] = np.nan
df.loc[df[col_map['jam_belajar']] > 12, col_map['jam_belajar']] = np.nan

df[col_map['jam_tidur']] = df[col_map['jam_tidur']].fillna(df[col_map['jam_tidur']].median()).clip(0, 12)
df[col_map['screen_time']] = df[col_map['screen_time']].fillna(df[col_map['screen_time']].median()).clip(0, 12)
df[col_map['jam_belajar']] = df[col_map['jam_belajar']].fillna(df[col_map['jam_belajar']].median()).clip(0, 12)

In [27]:
# ============================================================
# 4. SKORING & KOMPOSISI
# ============================================================
def get_edu_score(text):
    t = str(text).lower()
    if 'sarjana' in t or 's1' in t or 'master' in t or 'doctor' in t: return 5
    if 'diploma' in t: return 4
    if 'sma' in t: return 3
    if 'smp' in t: return 2
    if 'sd' in t: return 1
    return 3 # Default tengah

def get_income_score(text):
    pend_raw = str(text).strip().lower()
    if 'kurang dari' in pend_raw or '<' in pend_raw: return 1
    elif '2.000.000' in pend_raw: return 2
    elif '5.000.000' in pend_raw: return 3
    elif '> 10' in pend_raw or 'lebih' in pend_raw: return 4
    return 2 # Default

def normalize_text(t):
    if not isinstance(t, str): return ""
    return re.sub(r'\s+', ' ', t.lower()).replace('\n', ' ').strip()

def get_score(ans, a_word, b_word, c_word, a_val, b_val, c_val):
    t = normalize_text(ans)
    if a_word in t: return a_val
    if b_word in t: return b_val
    if c_word in t: return c_val
    return (a_val+b_val+c_val)/3

output = []
for idx, row in df.iterrows():
    nisn = str(row[col_map['nisn']]).strip()
    nama = str(row[col_map['nama']]).strip()
    email = str(row[col_map['email']]).strip() if 'email' in col_map else ''
    
    gender = 'Laki-laki' if 'laki' in str(row[col_map['gender']]).strip().lower() else 'Perempuan'
    
    jurusan_raw = str(row[col_map['jurusan']]).upper()
    if 'TKJ' in jurusan_raw: jurusan = 'TKJ'
    elif 'TPTU' in jurusan_raw: jurusan = 'TPTU'
    elif 'TAV' in jurusan_raw: jurusan = 'TAV'
    elif 'MULTI' in jurusan_raw: jurusan = 'MULTIMEDIA'
    else: jurusan = 'OTHER'
    
    jam_belajar = clean_num(row[col_map['jam_belajar']])
    jam_tidur = clean_num(row[col_map['jam_tidur']])
    screen_time = clean_num(row[col_map['screen_time']])
    nilai_raport = clean_num(row[col_map['nilai_raport']])
    kehadiran = clean_num(row[col_map['kehadiran']])
    
    # Advanced Features
    deviasi_tidur = abs(8 - jam_tidur)
    
    ayah_score = get_edu_score(row[col_map['pend_ayah']])
    ibu_score = get_edu_score(row[col_map['pend_ibu']])
    inc_score = get_income_score(row[col_map['pendapatan']])
    ses_index = ayah_score + ibu_score + inc_score # Range: 3 to 14
    
    kerja_sampingan = 'Ya' if 'ya' in str(row[col_map['kerja_sampingan']]).lower() else 'Tidak'
    
    komp_raw = str(row[col_map['kompetensi']]).lower()
    if 'sangat baik' in komp_raw or 'tinggi' in komp_raw: kompetensi = 'Tinggi'
    elif 'cukup' in komp_raw or 'sedang' in komp_raw: kompetensi = 'Menengah'
    else: kompetensi = 'Rendah'
    
    q9 = get_score(row[col_map['q9']], 'segera melepaskan', 'pesan error', 'cara cepat', 30, 50, 10)
    q10 = get_score(row[col_map['q10']], 'tutorial dasar', 'di sekolah saja', 'mencontohkan langkah', 50, 10, 30)
    ind_readiness = 'Siap' if (q9 + q10) > 50 else 'Belum Siap'
    
    q11 = get_score(row[col_map['q11']], 'ruang keluarga', 'sudut ruangan', 'kasur', 30, 50, 10)
    q12 = get_score(row[col_map['q12']], 'menghadap ke atas', 'dalam laci', 'dibalik', 10, 50, 30)
    env_score = q11 + q12
    study_env = 'Kondusif' if env_score >= 80 else 'Cukup Kondusif' if env_score >= 50 else 'Kurang Kondusif'
    
    q13 = get_score(row[col_map['q13']], 'sekarang juga', 'memaksakan diri', 'buku catatan', 10, 30, 50)
    q14 = get_score(row[col_map['q14']], 'multitasking', 'membalas obrolan', 'dijauhkan', 10, 30, 50)
    time_mgmt = q13 + q14
    
    q15 = get_score(row[col_map['q15']], 'tunda', 'finansial', 'sensasi', 20, 60, 100)
    motivasi = q15
    
    q16 = get_score(row[col_map['q16']], 'toilet', 'menyiapkan alat', 'acak', 35, 10, 20)
    q17 = get_score(row[col_map['q17']], 'mengiyakan', 'simulasi tekanan', 'berlebihan', 20, 10, 35)
    q18 = get_score(row[col_map['q18']], 'menghibur diri', 'salah jurusan', 'referensi', 15, 30, 5)
    stress_sum = q16 + q17 + q18
    stress_level = 'Berat' if stress_sum >= 80 else 'Sedang' if stress_sum >= 50 else 'Rendah'
    
    output.append({
        'nisn': nisn,
        'nama_siswa': nama,
        'email' : email,
        'gender': gender,
        'jurusan': jurusan,
        'kerja_sampingan': kerja_sampingan,
        'ses_index': ses_index,
        'jam_belajar_per_hari': jam_belajar,
        'screen_time': screen_time,
        'jam_tidur': jam_tidur,
        'deviasi_tidur': deviasi_tidur,
        'presentase_kehadiran': kehadiran,
        'industry_readiness': ind_readiness,
        'study_environment': study_env,
        'skor_time_management': time_mgmt,
        'motivasi_akademik': motivasi,
        'kompetensi_skill_level': kompetensi,
        'stress_level': stress_level,
        'nilai_rata_rata_raport': nilai_raport
    })

df_out = pd.DataFrame(output)

In [28]:
# ============================================================
# 5. Z-SCORE NORMALIZATION PER JURUSAN
# ============================================================
df_out['nilai_raport_zscore_jurusan'] = df_out.groupby('jurusan')['nilai_rata_rata_raport'].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0
)

# Fill NaNs jika ada (karena div by zero atau kelompok jurusan n=1)
df_out['nilai_raport_zscore_jurusan'] = df_out['nilai_raport_zscore_jurusan'].fillna(0)

df_out.to_csv(OUTPUT_FILE, index=False)
print(f"Konversi Selesai! Output disimpan ke {OUTPUT_FILE}")
print(f"Bentuk data: {df_out.shape[0]} baris x {df_out.shape[1]} kolom")


Konversi Selesai! Output disimpan ke d:\Pribadi\my project web\VOCAVISION\backend\ml_model\data_advanced.csv
Bentuk data: 199 baris x 20 kolom
